In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import BaggingRegressor


In [3]:
data = fetch_california_housing()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [5]:
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)

rmse_dt = np.sqrt(mean_squared_error(y_test, y_pred_dt))
r2_dt = r2_score(y_test, y_pred_dt)

print("Decision Tree RMSE:", rmse_dt)
print("Decision Tree R2:", r2_dt)

Decision Tree RMSE: 0.7037294974840077
Decision Tree R2: 0.622075845135081


In [6]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_dt = cross_val_score(dt, X, y, cv=cv, scoring='neg_mean_squared_error')

rmse_cv_dt = np.sqrt(-cv_dt)
print("Decision Tree CV RMSE:", rmse_cv_dt)
print("Mean:", rmse_cv_dt.mean(), "Std:", rmse_cv_dt.std())

Decision Tree CV RMSE: [0.7037295  0.7285791  0.71856166 0.70812042 0.71679828]
Mean: 0.7151577911748493 Std: 0.008654935817415292


# Bagging Regressor

In [7]:
bag = BaggingRegressor(
    estimator=DecisionTreeRegressor(),
    n_estimators=50,
    max_samples=0.8,
    max_features=1.0,
    bootstrap=True,
    random_state=42
)

bag.fit(X_train, y_train)
y_pred_bag = bag.predict(X_test)

rmse_bag = np.sqrt(mean_squared_error(y_test, y_pred_bag))
r2_bag = r2_score(y_test, y_pred_bag)

print("\nBagging RMSE:", rmse_bag)
print("Bagging R2:", r2_bag)


Bagging RMSE: 0.5140622547417071
Bagging R2: 0.7983377661950455


In [8]:
cv_bag = cross_val_score(bag, X, y, cv=cv, scoring='neg_mean_squared_error')
rmse_cv_bag = np.sqrt(-cv_bag)

print("Bagging CV RMSE:", rmse_cv_bag)
print("Mean:", rmse_cv_bag.mean(), "Std:", rmse_cv_bag.std())

Bagging CV RMSE: [0.51159208 0.51034682 0.51370223 0.48884704 0.52186236]
Mean: 0.5092701041958829 Std: 0.010971428683215538


In [9]:
comparison = pd.DataFrame({
    "Model": ["Decision Tree", "Bagging"],
    "CV Mean RMSE": [rmse_cv_dt.mean(), rmse_cv_bag.mean()],
    "CV Std Dev": [rmse_cv_dt.std(), rmse_cv_bag.std()]
})

print("\nStability Comparison:\n", comparison)



Stability Comparison:
            Model  CV Mean RMSE  CV Std Dev
0  Decision Tree      0.715158    0.008655
1        Bagging      0.509270    0.010971
